In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob

import pickle
from openai import OpenAI

from tenacity import (
                        retry,
                        stop_after_attempt,
                        wait_random_exponential
)

from tqdm import tqdm

In [ ]:
def generate_prompt_cta_prediction(description, cta_types):

    system_msg = f"""
           Your task is to classify a column's description of a given table with only one of the following classes that are separated by commas: {",".join(cta_types)}.

            Your instructions are:
                1. Look at the input given to you.
                2. Look at the description semantics in detail.
                3. Predict a class among the following classes {",".join(cta_types)} for the target column.
                4. Use "other" only when no available class provides a suitable semantic match.
                5. Answer only with the predicted class."""
            
    user_msg =  f"""Description: {description}
                    Class: """.strip()
    
    return system_msg, user_msg

In [ ]:
def generate_predictions(description, cta_types, client):
    
    system_msg_predict_cta, user_msg_predict_cta = generate_prompt_cta_prediction(description, cta_types)
    result = execute_prompt(client, system_msg_predict_cta, user_msg_predict_cta)
    cta = result.choices[0].message.content.split('Class: ')[-1].strip()
    
    return cta

In [ ]:
@retry(wait=wait_random_exponential(min=1,max=60), stop=stop_after_attempt(6))
def execute_prompt(client, system_msg, user_msg):
    completion = client.chat.completions.create(
                                            model="gpt-4o",
                                            messages=[
                                                        {
                                                            "role": "system", 
                                                             "content": f"{system_msg}"
                                                        },
                                                        {
                                                            "role": "user",
                                                            "content": f"{user_msg}"
                                                        }
                                                    ]
                                            )
    return completion

In [ ]:
client = OpenAI()

In [ ]:
BENCHMARK = 'NextiaJD-S' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider
descriptions = pd.read_csv(f'{BENCHMARK}/column_descriptions.csv')
descriptions.head()

In [ ]:
descriptions.shape

In [ ]:
cta_types = np.load('cta_types.npy')

In [ ]:
cta = []
cont, step = 0, 500

for table_name, column_name, description in tqdm(descriptions.values):
    target_description = description
    cta.append(generate_predictions(target_description, cta_types, client))
    
    cont += 1
    if(cont % step == 0):
        np.save(f'{BENCHMARK}/predicted_cta.npy', cta)
        
np.save(f'{BENCHMARK}/predicted_cta.npy', cta)

In [ ]:
descriptions['CTA'] = cta

In [ ]:
descriptions.to_csv(f'{BENCHMARK}/column_descriptions_with_cta.csv',index=False)